In [ ]:
from typing import Dict, List, Any
import requests
import math


In [ ]:
def fetch_forecast(lat: float, lon: float, models: List[str]) -> Dict[str, List[Dict[str, Any]]]:
    """Fetch 48 hour wind forecasts for several weather models.

    Args:
        lat: Latitude in decimal degrees.
        lon: Longitude in decimal degrees.
        models: List of model identifiers supported by Open-Meteo.

    Returns:
        Mapping of each model name to a list of hourly records with time,
        wind speed, gust and direction. Values may be None if data is
        missing from the API.
    """
    result: Dict[str, List[Dict[str, Any]]] = {}
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "windspeed_10m,windgusts_10m,winddirection_10m",
        "forecast_days": 2,
        "windspeed_unit": "kn",
    }
    for model in models:
        params["models"] = model
        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()
        hourly = response.json()["hourly"]
        records: List[Dict[str, Any]] = []
        for i, ts in enumerate(hourly["time"][0:48]):
            records.append({
                "time": ts,
                "wind": hourly["windspeed_10m"][i],
                "gust": hourly["windgusts_10m"][i],
                "direction": hourly["winddirection_10m"][i],
            })
        result[model] = records
    return result


In [ ]:
def find_first_valid_record(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Return first record containing both wind and direction values.

    Args:
        records: Sequence of forecast entries.

    Returns:
        The first item with non-None wind and direction, or the original
        first record if none match.
    """
    for record in records:
        if record.get("wind") is not None and record.get("direction") is not None:
            return record
    return records[0]


In [ ]:
def speed_to_color(speed: float) -> str:
    """Map wind speed to a simple blue→red color scale.

    Speeds of 0 knots are blue and 60 knots or greater are red.
    """
    ratio = min(max(speed / 60.0, 0.0), 1.0)
    red = int(255 * ratio)
    blue = int(255 * (1 - ratio))
    return f"#{red:02x}00{blue:02x}"


In [ ]:
def wind_barb_paths(cx: float, cy: float, direction: float, speed: float, length: float, stroke: str) -> List[str]:
    """Create SVG elements representing a meteorological wind barb.

    Angle is given in meteorological degrees, where 0 is north and values
    increase clockwise.
    """
    angle = math.radians(270 - direction)
    end_x = cx + length * math.cos(angle)
    end_y = cy + length * math.sin(angle)
    elems: List[str] = [f'<line x1="{cx}" y1="{cy}" x2="{end_x}" y2="{end_y}" stroke="{stroke}" stroke-width="2" />']
    barb_angle = angle + math.pi / 2
    spacing = 10
    x, y = end_x, end_y
    speed_kn = int(round(speed))

    def add_barb(x0: float, y0: float, size: float) -> None:
        x1 = x0 + size * math.cos(barb_angle)
        y1 = y0 + size * math.sin(barb_angle)
        elems.append(f'<line x1="{x0}" y1="{y0}" x2="{x1}" y2="{y1}" stroke="{stroke}" stroke-width="2" />')
    while speed_kn >= 50:
        x2 = x - spacing * math.cos(angle)
        y2 = y - spacing * math.sin(angle)
        x3 = x2 + 12 * math.cos(barb_angle)
        y3 = y2 + 12 * math.sin(barb_angle)
        elems.append(f'<polygon points="{x},{y} {x2},{y2} {x3},{y3}" fill="{stroke}" />')
        x, y = x2, y2
        speed_kn -= 50
    while speed_kn >= 10:
        add_barb(x, y, 10)
        x -= spacing * math.cos(angle)
        y -= spacing * math.sin(angle)
        speed_kn -= 10
    while speed_kn >= 5:
        add_barb(x, y, 5)
        x -= spacing * math.cos(angle)
        y -= spacing * math.sin(angle)
        speed_kn -= 5
    return elems


In [ ]:
def create_wind_map(lat: float, lon: float, direction: float | None, speed: float | None, output_path: str, size: int = 200) -> str:
    """Render a minimal map with a meteorological wind barb.

    Latitude and longitude are not yet used but kept for future map enhancements.
    """
    speed_val = float(speed) if speed is not None else 0.0
    dir_val = float(direction) if direction is not None else 0.0
    center = size // 2
    stroke = speed_to_color(speed_val)
    length = size // 3
    elements: List[str] = [
        f'<rect width="{size}" height="{size}" fill="#e6f2ff" />',
        f'<line x1="0" y1="{center}" x2="{size}" y2="{center}" stroke="#888" />',
    ]
    elements.extend(wind_barb_paths(center, center, dir_val, speed_val, length, stroke))
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{size}" height="{size}">'
        + "".join(elements)
        + "</svg>"
    )
    with open(output_path, "w", encoding="utf-8") as file:
        file.write(svg)
    return output_path


In [ ]:
def render_forecast_table(forecasts: Dict[str, List[Dict[str, Any]]], output_path: str) -> str:
    """Write forecast data to an SVG file containing plain text rows.

    The table lists time, wind and gust values for each model. Missing numbers are shown as a dash.
    """
    model_names = list(forecasts.keys())
    header = ["time"]
    for name in model_names:
        header.extend([f"{name} wind", f"{name} gust"])
    lines = [" ".join(header)]
    hours = len(next(iter(forecasts.values())))
    for i in range(min(48, hours)):
        row = [forecasts[model_names[0]][i]["time"]]
        for name in model_names:
            wind = forecasts[name][i]["wind"]
            gust = forecasts[name][i]["gust"]
            row.append(str(wind) if wind is not None else "-")
            row.append(str(gust) if gust is not None else "-")
        lines.append(" ".join(row))
    svg_lines = []
    for idx, text in enumerate(lines, start=1):
        svg_lines.append(f'<text x="0" y="{15 * idx}" font-size="12">{text}</text>')
    height = 15 * (len(lines) + 1)
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="900" height="{height}">'
        + "".join(svg_lines)
        + "</svg>"
    )
    with open(output_path, "w", encoding="utf-8") as file:
        file.write(svg)
    return output_path
